In [ ]:
# Load or reload R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Engineered Feature Distribution Visualizations (`models/plot_engineered.ipynb`)

This notebook constructs and visualizes the distributions of **26 Feature-Engineered Inputs** across the 5 Emergency Severity Index (ESI) Triage Classes (`1`, `2`, `3`, `4`, `5`):

### 10 Clinical Binary Indicator Features (Bar Plot Proportions):
1. `is_dyspnea_total`: `triage_vital_o2 < 90`
2. `is_dyspnea_moderate`: `90 < triage_vital_o2 < 94`
3. `is_bradypnea`: `triage_vital_rr < 10`
4. `is_tachypnea`: `triage_vital_rr > 30`
5. `is_hypotension`: `triage_vital_sbp <= 90`
6. `is_hypertension`: `triage_vital_sbp > 220`
7. `is_bradycardia_total`: `triage_vital_hr < 40`
8. `is_bradycardia_moderate`: `40 < triage_vital_hr < 60`
9. `is_tachycardia_total`: `triage_vital_hr > 150`
10. `is_tachycardia_moderate`: `100 < triage_vital_hr < 150`

### 16 Vital Delta & Range Features (Boxplots / Violins across ESI 1..5):
- `hr_mean_to_last`, `sbp_mean_to_last`, `spo2_mean_to_last`, `rr_mean_to_last`
- `hr_range`, `rr_range`, `spo2_range`, `sbp_range`
- `hr_last_to_min`, `sbp_last_to_min`, `spo2_last_to_min`, `rr_last_to_min`
- `hr_last_to_max`, `sbp_last_to_max`, `spo2_last_to_max`, `rr_last_to_max`

High-resolution PNG plots are saved to `plots/engineered_feature_distributions/`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Data & Construct All 26 Engineered Features in R
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last"); p_max    <- get_vec("pulse_max"); p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last");   s_max    <- get_vec("sbp_max");   s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last");  o2_max   <- get_vec("spo2_max");  o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last");   r_max    <- get_vec("resp_max");   r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr"); t_sbp <- get_vec("triage_vital_sbp"); t_o2 <- get_vec("triage_vital_o2"); t_rr <- get_vec("triage_vital_rr")
df_eng <- data.frame(
  esi                    = factor(as.character(raw_df[[target_col_name]]), levels = c("1", "2", "3", "4", "5")),
  is_dyspnea_total       = ifelse(t_o2 < 90, 1, 0),
  is_dyspnea_moderate    = ifelse(t_o2 > 90 & t_o2 < 94, 1, 0),
  is_bradypnea           = ifelse(t_rr < 10, 1, 0),
  is_tachypnea           = ifelse(t_rr > 30, 1, 0),
  is_hypotension         = ifelse(t_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(t_sbp > 220, 1, 0),
  is_bradycardia_total   = ifelse(t_hr < 40, 1, 0),
  is_bradycardia_moderate= ifelse(t_hr > 40 & t_hr < 60, 1, 0),
  is_tachycardia_total   = ifelse(t_hr > 150, 1, 0),
  is_tachycardia_moderate= ifelse(t_hr > 100 & t_hr < 150, 1, 0),
  hr_mean_to_last        = t_hr - p_last,
  sbp_mean_to_last       = t_sbp - s_last,
  spo2_mean_to_last      = t_o2 - o2_last,
  rr_mean_to_last        = t_rr - r_last,
  hr_range               = p_max - p_min,
  rr_range               = r_max - r_min,
  spo2_range             = o2_max - o2_min,
  sbp_range              = s_max - s_min,
  hr_last_to_min         = p_last - p_min,
  sbp_last_to_min        = s_last - s_min,
  spo2_last_to_min       = o2_last - o2_min,
  rr_last_to_min         = r_last - r_min,
  hr_last_to_max         = p_last - p_max,
  sbp_last_to_max        = s_last - s_max,
  spo2_last_to_max       = o2_last - o2_max,
  rr_last_to_max         = r_last - r_max
)
df_eng <- na.omit(df_eng)
df_eng_py <<- df_eng
cat(sprintf("Constructed 26 Engineered Features across %d rows\n", nrow(df_eng)))

In [ ]:
# ---------------------------------------------------------
# Step 2: Plot Distribution Graphs in Python
# ---------------------------------------------------------
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rpy2.robjects import r
import rpy2.robjects.pandas2ri as pandas2ri
try:
    pandas2ri.activate()
    df_eng = pd.DataFrame(pandas2ri.rpy2py_dataframe(r['df_eng_py']))
except Exception:
    df_eng = pd.DataFrame(r['df_eng_py'])
out_dir = "../plots/engineered_feature_distributions"
if not os.path.exists(out_dir): out_dir = "plots/engineered_feature_distributions"
os.makedirs(out_dir, exist_ok=True)
binary_feats = [
    'is_dyspnea_total', 'is_dyspnea_moderate', 'is_bradypnea', 'is_tachypnea',
    'is_hypotension', 'is_hypertension', 'is_bradycardia_total', 'is_bradycardia_moderate',
    'is_tachycardia_total', 'is_tachycardia_moderate'
]
continuous_feats = [
    'hr_mean_to_last', 'sbp_mean_to_last', 'spo2_mean_to_last', 'rr_mean_to_last',
    'hr_range', 'rr_range', 'spo2_range', 'sbp_range',
    'hr_last_to_min', 'sbp_last_to_min', 'spo2_last_to_min', 'rr_last_to_min',
    'hr_last_to_max', 'sbp_last_to_max', 'spo2_last_to_max', 'rr_last_to_max'
]
esi_colors = {'1': '#d62728', '2': '#ff7f0e', '3': '#1f77b4', '4': '#2ca02c', '5': '#9467bd'}
# 1. Plot Bar Charts for Binary Features (% positive by ESI Class)
fig, axes = plt.subplots(5, 2, figsize=(14, 18), dpi=200)
axes = axes.flatten()
for i, b_feat in enumerate(binary_feats):
    prop_df = df_eng.groupby('esi')[b_feat].mean().reset_index()
    prop_df['Percentage'] = prop_df[b_feat] * 100
    
    sns.barplot(data=prop_df, x='esi', y='Percentage', palette=esi_colors, ax=axes[i])
    axes[i].set_title(f'{b_feat} (% Positive Cases per ESI)', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('ESI Triage Level', fontsize=9)
    axes[i].set_ylabel('% Positive', fontsize=9)
    axes[i].grid(True, linestyle=':', alpha=0.5, axis='y')
    
    for p in axes[i].patches:
        axes[i].annotate(f'{p.get_height():.2f}%', (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='bottom', fontsize=8, xytext=(0, 2), textcoords='offset points')
plt.tight_layout()
bin_plot_path = os.path.join(out_dir, "binary_features_distribution.png")
plt.savefig(bin_plot_path)
plt.close()
print(f"Binary Features Distribution plot saved to: {bin_plot_path}")
# 2. Plot Boxplots for Continuous Features across ESI 1..5
fig, axes = plt.subplots(4, 4, figsize=(16, 14), dpi=200)
axes = axes.flatten()
for i, c_feat in enumerate(continuous_feats):
    sns.boxplot(data=df_eng, x='esi', y=c_feat, palette=esi_colors, ax=axes[i], showfliers=False)
    axes[i].set_title(f'{c_feat} across ESI', fontsize=10, fontweight='bold')
    axes[i].set_xlabel('ESI Level', fontsize=8)
    axes[i].set_ylabel('Value', fontsize=8)
    axes[i].grid(True, linestyle=':', alpha=0.5, axis='y')
plt.tight_layout()
cont_plot_path = os.path.join(out_dir, "continuous_features_distribution.png")
plt.savefig(cont_plot_path)
plt.close()
print(f"Continuous Features Distribution plot saved to: {cont_plot_path}")